# Team aEC/T Leaderboard

Rank UFA teams by possession aEC per throw. The main rate is `team_aec_per_throw = sum(total_aec) / sum(throw_count)`, which treats every throw as one unit in the team rate.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.fetching
import ufa_aec_possessions.leaderboards

importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.leaderboards)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    build_possessions,
    build_scoring_possessions,
    fetch_shownspace_season_throws_cached,
    filter_analysis_possessions,
    summarize_team_aec_consistency,
    summarize_team_aec_per_throw,
    summarize_team_top_possessions,
)

In [2]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'
MIN_POSSESSIONS = 20
TOP_TEAMS = 22
TOP_POSSESSION_COUNT = 5
TOP_POSSESSION_MIN_THROWS = 5
CONSISTENCY_TOP_N = 10
CONSISTENCY_MIN_THROWS = 5

## Load League Possessions

In [3]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_all_possessions, league_all_paths = build_possessions(league_throws)
league_possessions, league_paths = build_scoring_possessions(league_throws)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League all possessions found: {len(league_all_possessions):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')

League games loaded: 118
League throws loaded: 64,964
League all possessions found: 9,214
League scoring possessions found: 4,673


In [4]:
LEADERBOARD_COLUMNS = [
    'rank', 'team_id', 'team_aec_per_throw', 'avg_possession_aec_per_throw',
    'possessions', 'games', 'throws', 'total_aec', 'avg_throw_count',
    'avg_field_progress', 'huck_rate', 'resets_per_possession', 'o_line_share',
]

FORMATTERS = {
    'team_aec_per_throw': '{:.3f}',
    'avg_possession_aec_per_throw': '{:.3f}',
    'median_possession_aec_per_throw': '{:.3f}',
    'total_aec': '{:.1f}',
    'avg_throw_count': '{:.1f}',
    'avg_field_progress': '{:.1f}',
    'huck_rate': '{:.1%}',
    'resets_per_possession': '{:.1f}',
    'o_line_share': '{:.1%}',
    'top_mean_metric': '{:.3f}',
    'top_median_metric': '{:.3f}',
    'top_floor_metric': '{:.3f}',
    'top_ceiling_metric': '{:.3f}',
    'top_total_aec': '{:.1f}',
    'top_avg_throw_count': '{:.1f}',
    'median_metric': '{:.3f}',
    'p75_metric': '{:.3f}',
    'p90_metric': '{:.3f}',
    'top_n_mean_metric': '{:.3f}',
    'high_metric_share': '{:.1%}',
    'high_metric_threshold': '{:.3f}',
    'goal_share': '{:.1%}',
    'turnover_share': '{:.1%}',
}

TOP_POSSESSION_COLUMNS = [
    'rank', 'team_id', 'top_mean_metric', 'top_floor_metric', 'top_avg_throw_count',
    'team_aec_per_throw', 'median_possession_aec_per_throw', 'possessions', 'throws',
]

CONSISTENCY_COLUMNS = [
    'rank', 'team_id', 'median_metric', 'p75_metric', 'top_n_mean_metric',
    'high_metric_share', 'team_aec_per_throw', 'possessions', 'throws', 'avg_throw_count',
    'goal_share', 'turnover_share',
]


def show_team_aec_t_table(leaderboard, n=TOP_TEAMS, columns=LEADERBOARD_COLUMNS):
    table = leaderboard.head(n).reindex(columns=columns).copy()
    for column, formatter in FORMATTERS.items():
        if column in table:
            values = pd.to_numeric(table[column], errors='coerce')
            table[column] = values.map(lambda value: '' if pd.isna(value) else formatter.format(value))
    return table


POSITIVE_COLOR_SCALE = [
    [0.0, '#eaf4e8'],
    [0.35, '#9fd391'],
    [0.70, '#3f9c5a'],
    [1.0, '#09552f'],
]
DIVERGING_COLOR_SCALE = [
    [0.0, '#ba4a3c'],
    [0.50, '#f4f7f2'],
    [1.0, '#0b5d3b'],
]
HOVER_FORMATS = {
    'team_aec_per_throw': ':.3f',
    'avg_possession_aec_per_throw': ':.3f',
    'median_possession_aec_per_throw': ':.3f',
    'top_mean_metric': ':.3f',
    'top_floor_metric': ':.3f',
    'median_metric': ':.3f',
    'p75_metric': ':.3f',
    'top_n_mean_metric': ':.3f',
    'high_metric_share': ':.1%',
    'possessions': ':,',
    'games': ':,',
    'throws': ':,',
    'huck_rate': ':.1%',
    'o_line_share': ':.1%',
    'goal_share': ':.1%',
    'turnover_share': ':.1%',
}


def plot_team_metric_leaderboard(leaderboard, metric, title, x_label, n=TOP_TEAMS):
    chart = leaderboard.head(n).sort_values(metric, ascending=True).copy()
    if chart.empty:
        return px.bar(title=title)
    chart['team_label'] = chart['rank'].astype(int).astype(str) + '. ' + chart['team_id'].astype(str).str.title()
    hover_columns = [
        column for column in [
            'team_aec_per_throw', 'avg_possession_aec_per_throw', 'median_possession_aec_per_throw',
            'top_mean_metric', 'top_floor_metric', 'median_metric', 'p75_metric',
            'top_n_mean_metric', 'high_metric_share', 'possessions', 'games', 'throws',
            'huck_rate', 'o_line_share', 'goal_share', 'turnover_share',
        ]
        if column in chart and column != metric
    ]
    hover_data = {column: HOVER_FORMATS.get(column, True) for column in hover_columns}
    metric_min = pd.to_numeric(chart[metric], errors='coerce').min()
    metric_max = pd.to_numeric(chart[metric], errors='coerce').max()
    has_negative_values = metric_min < 0
    crosses_zero = metric_min < 0 < metric_max
    color_scale = DIVERGING_COLOR_SCALE if has_negative_values else POSITIVE_COLOR_SCALE
    padding = max((metric_max - metric_min) * 0.10, 0.015)
    x_range = [metric_min - padding, metric_max + padding] if has_negative_values else [0, metric_max + padding]
    fig = px.bar(
        chart,
        x=metric,
        y='team_label',
        orientation='h',
        text=metric,
        hover_name='team_label',
        hover_data=hover_data,
        labels={metric: x_label, 'team_label': 'Team'},
        color=metric,
        color_continuous_scale=color_scale,
    )
    fig.update_traces(
        texttemplate='%{x:.3f}',
        textposition='outside',
        cliponaxis=False,
        marker_line_color='rgba(255,255,255,0.85)',
        marker_line_width=0.8,
        opacity=0.96,
    )
    if crosses_zero:
        fig.add_vline(x=0, line_width=1, line_dash='dot', line_color='#6b7280')
        fig.update_coloraxes(cmid=0)
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'xanchor': 'center'},
        title_font={'size': 20, 'color': '#20385f'},
        height=max(560, 32 * len(chart) + 150),
        width=960,
        margin={'l': 135, 'r': 72, 't': 78, 'b': 62},
        coloraxis_showscale=False,
        bargap=0.24,
        font={'family': 'Segoe UI, Arial, sans-serif', 'size': 12, 'color': '#23395d'},
        hoverlabel={'bgcolor': '#ffffff', 'bordercolor': '#c9d8c4', 'font_size': 12, 'font_family': 'Segoe UI, Arial, sans-serif'},
        plot_bgcolor='#f7fbf5',
        paper_bgcolor='#ffffff',
    )
    fig.update_xaxes(
        title_text=x_label,
        range=x_range,
        showgrid=True,
        gridcolor='#e5eee2',
        zeroline=False,
        linecolor='#d7e4d2',
        tickfont={'color': '#31476b'},
        title_font={'color': '#31476b'},
    )
    fig.update_yaxes(
        title_text='',
        showgrid=False,
        ticks='',
        tickfont={'color': '#20385f'},
    )
    return fig


def plot_team_aec_t_leaderboard(leaderboard, title, n=TOP_TEAMS):
    return plot_team_metric_leaderboard(
        leaderboard,
        metric='team_aec_per_throw',
        title=title,
        x_label='team aEC/T',
        n=n,
    )

## All Possessions Including Turnovers

This is the closest view to Shown Space team `Tot-aEC`: it includes scoring possessions, turnovers, and other built possessions. This is the view where Bighorns show up as negative overall.

In [5]:
team_aec_t_all_possessions = summarize_team_aec_per_throw(
    league_all_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_possessions,
    title=f'{SEASON} UFA team aEC/T - all possessions including turnovers',
)

In [6]:
show_team_aec_t_table(team_aec_t_all_possessions)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,empire,0.072,0.109,408,11,3133,226.9,7.7,62.9,34.1%,2.0,58.6%
1,2,flyers,0.066,0.086,474,12,3467,227.5,7.3,58.9,36.5%,1.9,58.2%
2,3,windchill,0.065,0.091,404,11,3062,199.2,7.6,61.4,36.9%,2.2,62.6%
3,4,breeze,0.060,0.046,431,11,2871,173.0,6.7,64.1,35.0%,1.5,67.3%
4,5,spiders,0.060,0.068,395,10,2890,172.2,7.3,53.4,22.0%,1.7,55.9%
5,6,hustle,0.057,0.067,415,11,3233,184.5,7.8,61.7,35.2%,2.1,65.8%
6,7,sol,0.055,0.062,410,10,3036,167.4,7.4,57.6,28.8%,1.9,62.4%
7,8,growlers,0.053,0.070,429,11,2896,152.4,6.8,55.3,35.0%,1.5,69.7%
8,9,glory,0.050,0.052,353,10,3098,154.6,8.8,58.2,27.8%,2.5,62.0%
9,10,shred,0.049,0.038,445,11,3084,150.7,6.9,58.4,33.3%,1.8,62.7%


## Total aEC Sanity Check

Shown Space's `Tot-aEC` is a total, not an aEC/T rate. Sorting the all-possession summary by `total_aec` should line up with that table much more closely than the scoring-only views.

In [7]:
team_total_aec_low_to_high = team_aec_t_all_possessions.sort_values('total_aec', ascending=True).reset_index(drop=True).copy()
team_total_aec_low_to_high['rank'] = range(1, len(team_total_aec_low_to_high) + 1)
show_team_aec_t_table(team_total_aec_low_to_high)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,bighorns,-0.021,-0.119,497,11,2521,-53.0,5.1,46.9,37.4%,1.4,82.5%
1,2,havoc,0.007,-0.054,408,10,2917,19.7,7.1,53.9,38.2%,2.0,82.6%
2,3,thunderbirds,0.019,-0.046,414,10,2787,54.0,6.7,58.7,41.1%,2.0,80.9%
3,4,steel,0.020,-0.016,448,11,2876,58.4,6.4,56.1,39.1%,1.6,84.4%
4,5,phoenix,0.025,-0.026,435,11,3034,76.6,7.0,62.4,41.8%,1.7,80.0%
5,6,union,0.026,-0.018,357,10,2963,77.6,8.3,62.5,35.0%,2.3,77.6%
6,7,apex,0.032,-0.004,434,11,2846,92.1,6.6,63.4,37.1%,1.3,82.0%
7,8,rush,0.044,0.038,461,11,2849,125.3,6.2,59.2,39.7%,1.5,69.8%
8,9,royal,0.044,0.023,411,11,2902,126.6,7.1,59.2,33.6%,1.5,73.7%
9,10,radicals,0.048,0.031,370,10,2689,127.7,7.3,59.1,25.4%,1.8,70.0%


## All Scoring Possessions

This view only includes possessions that ended in goals. It answers a narrower question: when a team scores, how much aEC does that scoring possession create per throw? It should not be read as overall team aEC.

In [8]:
team_aec_t_all_scoring = summarize_team_aec_per_throw(
    league_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_all_scoring,
    title=f'{SEASON} UFA team aEC/T - all scoring possessions',
)

In [9]:
show_team_aec_t_table(team_aec_t_all_scoring)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,bighorns,0.153,0.287,144,11,930,142.3,6.5,72.2,59.7%,1.6,86.1%
1,2,rush,0.141,0.262,228,11,1613,227.9,7.1,70.5,46.9%,1.7,69.3%
2,3,steel,0.137,0.231,182,11,1329,182.7,7.3,73.2,54.9%,1.7,80.8%
3,4,growlers,0.133,0.269,205,11,1552,206.9,7.6,67.2,43.4%,1.7,69.3%
4,5,apex,0.129,0.210,191,11,1503,194.2,7.9,76.3,42.4%,1.6,78.0%
5,6,breeze,0.128,0.216,245,11,1912,244.4,7.8,73.8,33.5%,1.7,65.7%
6,7,shred,0.127,0.224,242,11,1908,243.2,7.9,72.1,40.1%,1.9,65.3%
7,8,alleycats,0.126,0.240,202,10,1615,203.2,8.0,74.6,52.0%,2.0,67.3%
8,9,cascades,0.125,0.236,212,11,1704,213.5,8.0,68.7,44.8%,1.8,68.4%
9,10,windchill,0.124,0.224,250,11,2024,250.9,8.1,66.3,38.0%,2.3,60.0%


## Top 5 Non-Huck Scoring Possessions

Raw top-five aEC/T can overrate one-throw or very short scores. This version excludes hucks and requires at least `TOP_POSSESSION_MIN_THROWS` throws before selecting each team's top five.

In [10]:
non_huck_scoring_possessions = league_possessions[
    pd.to_numeric(league_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()

team_top5_non_huck_aec_t = summarize_team_top_possessions(
    non_huck_scoring_possessions,
    top_n=TOP_POSSESSION_COUNT,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=TOP_POSSESSION_MIN_THROWS,
)

plot_team_metric_leaderboard(
    team_top5_non_huck_aec_t,
    metric='top_mean_metric',
    title=f'{SEASON} UFA top {TOP_POSSESSION_COUNT} non-huck scoring possession aEC/T, min {TOP_POSSESSION_MIN_THROWS} throws',
    x_label=f'top {TOP_POSSESSION_COUNT} mean aEC/T',
)

In [11]:
show_team_aec_t_table(team_top5_non_huck_aec_t, columns=TOP_POSSESSION_COLUMNS)

,rank,team_id,top_mean_metric,top_floor_metric,top_avg_throw_count,team_aec_per_throw,median_possession_aec_per_throw,possessions,throws
0,1,hustle,0.221,0.214,5.0,0.083,0.089,122,1485
1,2,empire,0.218,0.191,5.8,0.088,0.091,131,1501
2,3,windchill,0.216,0.210,5.0,0.092,0.100,119,1298
3,4,rush,0.212,0.199,5.0,0.092,0.099,80,877
4,5,flyers,0.211,0.200,5.0,0.084,0.093,146,1725
5,6,breeze,0.210,0.199,5.0,0.095,0.099,129,1360
6,7,royal,0.210,0.200,5.0,0.090,0.100,107,1189
7,8,steel,0.209,0.190,5.0,0.090,0.099,60,674
8,9,growlers,0.208,0.200,5.0,0.084,0.101,82,972
9,10,radicals,0.205,0.199,5.0,0.081,0.084,105,1292


## Consistent Non-Huck Scoring aEC/T

This is still scoring-only, so turnovers are not included. It asks which teams repeatedly finish successful non-huck scores with high aEC/T. The next section adds turnovers back in.

In [12]:
eligible_non_huck_scoring = non_huck_scoring_possessions[
    pd.to_numeric(non_huck_scoring_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_scoring['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_scoring_possessions,
    high_threshold=non_huck_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck scoring possessions: {len(eligible_non_huck_scoring):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck scoring aEC/T, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck scoring aEC/T',
)

Eligible non-huck scoring possessions: 2,163
High aEC/T threshold, 75th percentile: 0.127


In [13]:
show_team_aec_t_table(team_non_huck_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

,rank,team_id,median_metric,p75_metric,top_n_mean_metric,high_metric_share,team_aec_per_throw,possessions,throws,avg_throw_count,goal_share,turnover_share
0,1,bighorns,0.112,0.143,0.173,36.8%,0.097,38,381,10.0,100.0%,0.0%
1,2,growlers,0.101,0.128,0.202,25.6%,0.084,82,972,11.9,100.0%,0.0%
2,3,shred,0.101,0.138,0.198,33.0%,0.095,112,1197,10.7,100.0%,0.0%
3,4,spiders,0.101,0.126,0.192,24.0%,0.092,150,1635,10.9,100.0%,0.0%
4,5,apex,0.100,0.124,0.179,20.0%,0.094,90,982,10.9,100.0%,0.0%
5,6,windchill,0.100,0.143,0.208,29.4%,0.092,119,1298,10.9,100.0%,0.0%
6,7,royal,0.100,0.125,0.191,23.4%,0.090,107,1189,11.1,100.0%,0.0%
7,8,breeze,0.099,0.139,0.195,27.9%,0.095,129,1360,10.5,100.0%,0.0%
8,9,rush,0.099,0.130,0.193,27.5%,0.092,80,877,11.0,100.0%,0.0%
9,10,steel,0.099,0.133,0.185,30.0%,0.090,60,674,11.2,100.0%,0.0%


## Consistent Non-Huck aEC/T Including Turnovers

This uses all built possessions, filters out hucks, and includes turnovers. This is a better team-quality version of the consistency view because failed possessions count against the team.

In [14]:
non_huck_all_possessions = league_all_possessions[
    pd.to_numeric(league_all_possessions['huck_count'], errors='coerce').fillna(0).eq(0)
].copy()
eligible_non_huck_all = non_huck_all_possessions[
    pd.to_numeric(non_huck_all_possessions['throw_count'], errors='coerce').ge(CONSISTENCY_MIN_THROWS)
].copy()
non_huck_all_high_aec_t_threshold = pd.to_numeric(
    eligible_non_huck_all['aec_per_throw'],
    errors='coerce',
).quantile(0.75)

team_non_huck_all_aec_t_consistency = summarize_team_aec_consistency(
    non_huck_all_possessions,
    high_threshold=non_huck_all_high_aec_t_threshold,
    min_possessions=MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
)

print(f'Eligible non-huck all possessions: {len(eligible_non_huck_all):,}')
print(f'High aEC/T threshold, 75th percentile: {non_huck_all_high_aec_t_threshold:.3f}')

plot_team_metric_leaderboard(
    team_non_huck_all_aec_t_consistency,
    metric='median_metric',
    title=f'{SEASON} UFA consistent non-huck aEC/T including turnovers, min {CONSISTENCY_MIN_THROWS} throws',
    x_label='median non-huck all-possession aEC/T',
)

Eligible non-huck all possessions: 3,770
High aEC/T threshold, 75th percentile: 0.101


In [15]:
show_team_aec_t_table(team_non_huck_all_aec_t_consistency, columns=CONSISTENCY_COLUMNS)

,rank,team_id,median_metric,p75_metric,top_n_mean_metric,high_metric_share,team_aec_per_throw,possessions,throws,avg_throw_count,goal_share,turnover_share
0,1,spiders,0.083,0.117,0.192,35.7%,0.062,210,2178,10.4,71.4%,27.1%
1,2,breeze,0.080,0.121,0.195,33.2%,0.065,184,1828,9.9,70.1%,26.6%
2,3,windchill,0.080,0.113,0.208,32.8%,0.056,177,1865,10.5,67.2%,30.5%
3,4,empire,0.080,0.110,0.198,29.8%,0.062,181,1949,10.8,72.4%,24.9%
4,5,flyers,0.076,0.119,0.204,33.5%,0.063,191,2105,11.0,76.4%,22.0%
5,6,sol,0.068,0.111,0.185,28.2%,0.050,188,2024,10.8,64.4%,33.0%
6,7,hustle,0.064,0.103,0.212,26.5%,0.045,189,2192,11.6,64.6%,33.9%
7,8,royal,0.064,0.107,0.191,28.2%,0.047,177,1867,10.5,60.5%,39.0%
8,9,shred,0.064,0.108,0.198,28.4%,0.043,197,1953,9.9,56.9%,41.6%
9,10,glory,0.062,0.084,0.176,14.2%,0.042,183,2270,12.4,64.5%,33.3%


## Default Possession-Shape Pool

This uses the same default filter as the top-possession shape work: O-line scoring possessions, long-field starts, hucks excluded.

In [16]:
analysis_possessions, analysis_paths = filter_analysis_possessions(
    league_possessions,
    league_paths,
)
team_aec_t_default_pool = summarize_team_aec_per_throw(
    analysis_possessions,
    min_possessions=MIN_POSSESSIONS,
)

plot_team_aec_t_leaderboard(
    team_aec_t_default_pool,
    title=f'{SEASON} UFA team aEC/T - O-line non-huck long-field scores',
)

In [17]:
show_team_aec_t_table(team_aec_t_default_pool)

,rank,team_id,team_aec_per_throw,avg_possession_aec_per_throw,possessions,games,throws,total_aec,avg_throw_count,avg_field_progress,huck_rate,resets_per_possession,o_line_share
0,1,shred,0.095,0.109,68,11,732,69.7,10.8,86.8,0.0%,2.5,100.0%
1,2,bighorns,0.095,0.115,30,10,303,28.8,10.1,82.8,0.0%,2.5,100.0%
2,3,phoenix,0.094,0.116,60,11,645,60.8,10.8,85.5,0.0%,2.4,100.0%
3,4,spiders,0.091,0.106,78,10,873,79.8,11.2,82.8,0.0%,2.2,100.0%
4,5,breeze,0.091,0.106,85,11,949,86.3,11.2,88.7,0.0%,2.4,100.0%
5,6,empire,0.090,0.110,81,11,934,83.8,11.5,89.9,0.0%,2.7,100.0%
6,7,royal,0.089,0.105,71,11,791,70.7,11.1,81.2,0.0%,2.5,100.0%
7,8,apex,0.089,0.100,59,11,675,60.0,11.4,88.5,0.0%,2.4,100.0%
8,9,windchill,0.088,0.104,55,11,642,56.4,11.7,85.1,0.0%,3.0,100.0%
9,10,rush,0.086,0.101,49,11,584,50.3,11.9,85.8,0.0%,2.8,100.0%
